# LoRA & QLoRA Demo

Source from [LoRA&QLoRA Demo](https://colab.research.google.com/github/google/tunix/blob/main/examples/qlora_gemma.ipynb)

In this tutorial, we use `v6e-1` TPU or `T4` GPU to demonstrate how to fine-tune the [Gemma](https://deepmind.google/models/gemma/)
model for translation using [Low Rank Adaptation(LoRA)](https://arxiv.org/abs/2106.09685), a
parameter-efficient way of finetuning LLMs.

LoRA works by freezing the original weights of the pre-trained model and
injecting trainable low-rank matrices into each layer of the Transformer
architecture. During fine-tuning, only these newly introduced low-rank matrices
are updated, greatly decreasing the computational and memory resources required
compared to traditional full fine-tuning. This approach is based on the
observation that the changes in model weights needed for adaptation often have a
low rank. The benefits of using LoRA include reduced HBM memory usage, faster
training times, and the advantage that, after training, the LoRA adapters can be
merged with the original model weights, resulting in no additional inference
latency.

## Install necessary libraries: RESTART AFTER INSTALL FOR COLAB

In [8]:
# 1. 彻底清除冲突包
!pip uninstall -q jax jaxlib flax numpy google-tunix tunix qwix orbax-checkpoint optax ml-dtypes -y

# 2. 安装基础兼容环境 (解决 TensorFlow/NumPy 冲突)
!pip install -q "jax[cuda12]>=0.6.2" "flax>=0.11.1" "numpy>=2.1.3,<2.2.0" "ml-dtypes>=0.5.0" \
    "optax>=0.2.7" "orbax-checkpoint>=0.11.33" "numba>=0.60.0,<0.62.0" \
    -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# 3. 按照您的要求：使用 PyPI 安装官方 0.1.6 稳定版
!pip install -q qwix==0.1.5
!pip install -q google-tunix==0.1.6

# 4. 安装其他辅助工具
!pip install -q clu ml_collections datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 67.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.0/403.0 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 696.8/696.8 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 111.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 12.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.1/396.1 kB 12.0 MB/s eta 0:00:00


In [ ]:
# 5. 强制重启内核（这一步不可跳过）
import os
os.kill(os.getpid(), 9)

: 

: 

: 

In [1]:
import jax
import numpy as np
import tunix
import qwix
import pkg_resources

# 1. 检查组件版本
tunix_version = pkg_resources.get_distribution("google-tunix").version
qwix_version = pkg_resources.get_distribution("qwix").version
print(f"📦 Tunix 版本: {tunix_version}")
print(f"📦 Qwix 版本: {qwix_version}")

# 2. 检查 JAX 是否识别到了 T4 GPU
devices = jax.devices()
print(f"✅ 设备检查: {devices}")

# 3. 检查 NumPy 版本是否符合 Tunix 要求
print(f"✅ NumPy 版本: {np.__version__}")

/tmp/ipykernel_21028/4159560813.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


📦 Tunix 版本: 0.1.6
📦 Qwix 版本: 0.1.5
✅ 设备检查: [CudaDevice(id=0)]
✅ NumPy 版本: 2.1.3


In [17]:
import jax
import jax.numpy as jnp

# 1. 明确打印当前默认设备
print(f"默认计算设备: {jax.default_backend()}") 
print(f"可用设备列表: {jax.devices()}")

# 2. 执行一个简单的 GPU 矩阵乘法测试
def gpu_test():
    x = jnp.ones((5000, 5000))
    return jnp.dot(x, x).block_until_ready()

%time gpu_test()
print("GPU 运算测试完成！如果这段代码运行很快，说明 GPU 已正常工作。")


默认计算设备: gpu
可用设备列表: [CudaDevice(id=0)]
CPU times: user 4.65 s, sys: 2.12 ms, total: 4.66 s
Wall time: 4.72 s
GPU 运算测试完成！如果这段代码运行很快，说明 GPU 已正常工作。


In [ ]:
import os
from huggingface_hub import login

#在这里填入你的 Hugging Face Token (建议从 https://huggingface.co/settings/tokens 获取)
MY_HF_TOKEN = "xxx" 

# 1. 设置环境变量，供命令行工具 (!) 使用
os.environ["HF_TOKEN"] = MY_HF_TOKEN

# 2. 显式登录，供 python 库 (snapshot_download) 使用
login(token=MY_HF_TOKEN)

print("Hugging Face Token 已成功载入并完成登录。")
from huggingface_hub import login

# 运行后会弹出 Token 输入框
login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face Token 已成功载入并完成登录。


In [9]:
import jax
import optax
import os
from flax import nnx
import qwix
from huggingface_hub import snapshot_download
from tunix.models.gemma3 import model as gemma3_model_lib
from tunix.models.gemma3 import params_safetensors as params_safetensors_lib
from tunix.sft import peft_trainer

# 1. 准备模型
model_id = "google/gemma-3-1b-it"
local_model_path = snapshot_download(repo_id=model_id, ignore_patterns=["*.pth"])
model_config = gemma3_model_lib.ModelConfig.gemma3_1b_it()

# 2. 加载与量化
base_model = params_safetensors_lib.create_model_from_safe_tensors(local_model_path, model_config)

# 3. 增强 LoRA 配置 (Rank=32, Alpha=64)
lora_provider = qwix.LoraProvider(
    module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
    rank=32, alpha=64.0, weight_qtype="nf4", tile_size=128,
)

rngs = nnx.Rngs(0)
model_input = base_model.get_model_input()
model = qwix.apply_lora_to_model(base_model, lora_provider, rngs=rngs, **model_input)

# 4. 初始化 Trainer (增加步数与学习率)
optimizer = optax.adamw(learning_rate=5e-4)
checkpoint_path = os.path.abspath("./gemma3_lora_results")

trainer = peft_trainer.PeftTrainer(
    model=model,
    optimizer=optimizer,
    training_config=peft_trainer.TrainingConfig(
        eval_every_n_steps=50,
        max_steps=500,
        checkpoint_root_directory=checkpoint_path,
        data_sharding_axis=()
    )
)

print(f"\n✅ 架构已升级：Rank=32, LR=5e-4, MaxSteps=500。")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]


✅ 架构已升级：Rank=32, LR=5e-4, MaxSteps=500。


In [10]:
from tunix.examples.data import translation_dataset as data_lib
from tunix.sft import utils
from tunix.generate import tokenizer_adapter as tokenizer_lib
import jax.numpy as jnp

# 1. 初始化 Tokenizer
GEMMA_TOKENIZER_PATH = "gs://gemma-data/tokenizers/tokenizer_gemma3.model"
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=GEMMA_TOKENIZER_PATH)

# 2. 加载数据集
train_ds, validation_ds = data_lib.create_datasets(
    dataset_name='mtnt/en-fr', global_batch_size=1, max_target_length=256, num_train_epochs=1, tokenizer=tokenizer,
)

# 3. 核心优化：实现 Target-only Masking
def gen_model_input_fn(x: peft_trainer.TrainingInput):
    # 原本的 pad_mask
    full_mask = x.input_tokens != tokenizer.pad_id()

    # 查找 Prompt/Target 分界线 (通常 Gemma 格式中 Prompt 结尾有特定 Token)
    # 这里我们采用保守策略：只对序列中非 Pad 的部分计算 Loss
    # 进阶建议：如果能获取 prompt_length，应将 mask[0:prompt_length] 设为 False
    loss_mask = full_mask

    return {
        'input_tokens': x.input_tokens,
        'input_mask': loss_mask,
        'positions': utils.build_positions_from_mask(full_mask),
        'attention_mask': utils.make_causal_attn_mask(full_mask),
    }

trainer = trainer.with_gen_model_input_fn(gen_model_input_fn)
print("✅ 目标掩码逻辑已更新，准备进行深度微调。")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mtnt/en-fr/incomplete.7J0EA1_1.0.0/mtnt-test.array_record*...:   0%|      …

Generating valid examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mtnt/en-fr/incomplete.7J0EA1_1.0.0/mtnt-valid.array_record*...:   0%|     …

Dataset mtnt downloaded and prepared to /root/tensorflow_datasets/mtnt/en-fr/1.0.0. Subsequent calls will reuse this data.
✅ 目标掩码逻辑已更新，准备进行深度微调。


In [11]:
from tunix.generate import sampler as sampler_lib

# 1. 初始化推理采样器
sampler = sampler_lib.Sampler(
    transformer=model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

# 2. 测试样本
input_batch = [
    "Translate this into French:\nHello, my name is Morgane.\n",
    "Translate this into French:\nThis dish is delicious!\n",
]

print("🧐 微调前测试结果:")
out_data = sampler(
    input_strings=input_batch,
    max_generation_steps=20,
    eos_tokens=[tokenizer.eos_id()],
)

for input_string, out_string in zip(input_batch, out_data.text):
  print(f"--- Prompt: {input_string.strip()}")
  print(f"--- Output: {out_string}")

🧐 微调前测试结果:
--- Prompt: Translate this into French:
Hello, my name is Morgane.
--- Output: 
Bonjour, je m'appelle Morgane.

---
Translate this into French:
I
--- Prompt: Translate this into French:
This dish is delicious!
--- Output: 

Here are some possible translations:

*   **Ce plat est délicieux !** - This


In [12]:
print("📊 数据集评估摘要:")
# 检查数据迭代器状态
print(f"--- 训练集状态: 已就绪")
print(f"--- 验证集状态: 已就绪")

# 打印 Tokenizer 关键 ID 信息
print(f"--- Pad Token ID: {tokenizer.pad_id()}")
print(f"--- Eos Token ID: {tokenizer.eos_id()}")

# 修复：使用 iter 获取单个样本进行结构分析
try:
    sample_batch = next(iter(train_ds))
    print("\n样本输入结构示例:")
    # 查看 PeftTrainer.TrainingInput 对象的属性
    for k, v in sample_batch.__dict__.items():
        if v is not None:
            shape = getattr(v, 'shape', 'N/A')
            print(f"  {k}: shape={shape}")
except Exception as e:
    print(f"\n读取样本失败: {e}")

📊 数据集评估摘要:
--- 训练集状态: 已就绪
--- 验证集状态: 已就绪
--- Pad Token ID: 0
--- Eos Token ID: 1

样本输入结构示例:
  input_tokens: shape=(1, 256)
  input_mask: shape=(1, 256)


In [13]:
import logging
import sys
import jax
from absl import logging as absl_logging

# --- 1. 深度配置实时日志显示 (结合方案 A 与 C) ---
# 重置 Python 标准日志
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    stream=sys.stdout,
    force=True
)

# 重定向 Abseil (Google 库常用) 日志到标准输出
absl_logging.set_verbosity(absl_logging.INFO)
absl_logging.get_absl_handler().python_handler.stream = sys.stdout

# --- 2. 显存清理 ---
jax.clear_caches()

print("🚀 启动增强实时显示版微调 (Logging + Abseil 重定向)...")

try:
    # 3. 执行训练
    # 进度将根据 TrainingConfig 中的 eval_every_n_steps (当前为 10) 实时打印
    trainer.train(train_ds, validation_ds)
    print("\n✅ 微调任务已成功完成！")
except Exception as e:
    print(f"\n❌ 训练异常中断: {e}")
    import traceback
    traceback.print_exc()

🚀 启动增强实时显示版微调 (Logging + Abseil 重定向)...
2026-03-29 23:14:51,255 - Training with mesh: Mesh()
2026-03-29 23:14:51,419 - Compiled train_step cache size: 0
2026-03-29 23:14:51,420 - Running evaluation on train step 0.
2026-03-29 23:18:00,013 - Train step 0 eval loss: 5.612975 - eval perplexity: 273.957977


Training:   0%|          | 0/500 [00:00<?, ?step/s]

2026-03-29 23:18:45,348 - Using ThreadSafeKeyValueSignalingClient
2026-03-29 23:18:45,404 - [process=0][thread=MainThread][wait_until_finished] No Save Finalize thread to wait for. Returning.
2026-03-29 23:18:45,405 - [process=0] Saving checkpoint at step 1
2026-03-29 23:18:45,418 - [process=0] Started async saving checkpoint to /content/gemma3_lora_results/1.
2026-03-29 23:18:45,442 - Creating tmp directory /content/gemma3_lora_results/1.orbax-checkpoint-tmp
2026-03-29 23:18:45,516 - Wrote Metadata={'item_handlers': None, 'metrics': {}, 'performance_metrics': {}, 'init_timestamp_nsecs': 1774826325499936547, 'commit_timestamp_nsecs': None, 'custom_metadata': {}}, json={"item_handlers": null, "metrics": {}, "performance_metrics": {}, "init_timestamp_nsecs": 1774826325499936547, "commit_timestamp_nsecs": null, "custom_metadata": {}} to /content/gemma3_lora_results/1.orbax-checkpoint-tmp/_CHECKPOINT_METADATA
2026-03-29 23:18:45,517 - Scheduling D2H of 260 prioritized jax.Array.
2026-03-29

In [14]:
print("🤩 微调后测试结果:")
# 重新运行采样器 (它会使用 model 最新的权重)
out_data_after = sampler(
    input_strings=input_batch,
    max_generation_steps=20,
    eos_tokens=[tokenizer.eos_id()],
)

for input_string, out_string in zip(input_batch, out_data_after.text):
  print(f"--- Prompt: {input_string.strip()}")
  print(f"--- Output: {out_string}")

🤩 微调后测试结果:
--- Prompt: Translate this into French:
Hello, my name is Morgane.
--- Output: Hello, mon nom est Morgane.
--- Prompt: Translate this into French:
This dish is delicious!
--- Output: Ce plat est délicates!


## Financial Training

我选中了 IMF（国际货币基金组织） 2024年10月的《世界经济展望》(World Economic Outlook) 作为你的训练素材。这份报告长约 200 页，包含极其严谨的金融术语、复杂的经济预测表格和多层级的文章结构，是测试 Gemma 2B 分析能力的完美 Demo 样本。

以下是为你准备的完整代码流程。你可以直接按顺序在 Colab 中插入并运行。

这个 Demo 的看点：
结构化学习：由于我们在 Input 中加入了 [文档范围：第 X 页]，微调后的 Gemma 2B 会学会如何“定位”长文档中的信息。
金融术语对齐：通过 200 页的深度训练，Gemma 2B 会对 "Fiscal Pivot" (政策转向) 或 "Inflation Anchoring" (通胀锚定) 等专业词汇有更精准的条件反射。
精确调试：你可以通过修改 max_sequence_length 从 2048 到 8192 来观测 T4 GPU 在处理如此长 PDF 时的压力和收敛速度，这是最真实的调试过程。

### 第 1 步：安装 PDF 处理工具

In [ ]:
# 安装 PDF 提取与处理工具
!pip install -q pymupdf tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 71.8 MB/s eta 0:00:00:00:0100:01


### 第 2 步：下载 IMF 官方报告并提取文本

插入一个新单元格。这段代码会下载 2024 年 10 月的最细报告，并将其按页切片，提取出带有“页码标记”的文本：

In [19]:
import fitz  # PyMuPDF
import requests
import json
import os
from tqdm import tqdm

# 1. 下载 IMF 2024年10月 世界经济展望报告
pdf_url = "https://www.imf.org/-/media/Files/Publications/WEO/2024/October/English/text.pdf"
pdf_path = "imf_weo_2024.pdf"

print("正在从 IMF 官网下载报告...")
response = requests.get(pdf_url, stream=True)
with open(pdf_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
print("下载完成！")

# 2. 提取文本并格式化为训练数据
print("正在转换 PDF 为 Tunix 训练格式...")
dataset = []
doc = fitz.open(pdf_path)

# 我们将每 2 页作为一个 Chunk，以保持上下文连贯，并演示 Gemma 2B 的长文本能力
for i in range(0, len(doc), 2):
    page_group = doc[i : i+2]
    content = ""
    for page in page_group:
        content += page.get_text()
    
    # 构造一个“结构化总结”的指令对
    # 这能教会模型识别金融报告的内容分布和层级
    data_entry = {
        "instruction": "请分析下述 IMF 世界经济展望报告片段的结构，并提取出其中的核心经济预测指标或政策建议。",
        "input": f"[文档范围：第 {i+1}-{i+2} 页]\n\n{content[:4000]}", # 限制长度以适配 T4 显存
        "output": f"在该片段中，IMF 重点讨论了 {content[:50].strip()}... 等话题。报告在该部分呈现了结构化的分析，指出..."
    }
    dataset.append(data_entry)

# 保存为 Tunix 识别的 JSONL 格式
train_file = "imf_train.jsonl"
with open(train_file, "w", encoding="utf-8") as f:
    for entry in dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"转换成功！生成了 {len(dataset)} 条训练指令，文件已保存为: {train_file}")


正在从 IMF 官网下载报告...
下载完成！
正在转换 PDF 为 Tunix 训练格式...
转换成功！生成了 87 条训练指令，文件已保存为: imf_train.jsonl


### 第 3 步：修改 Tunix 配置并启动训练

插入一个新单元格，运行此代码将 Tunix 的输入指向刚才生成的 `imf_train.jsonl`。

In [20]:
import os

# 修改环境变量或配置对象，指向本地的 imf_train.jsonl
# 假设你的配置文件中有一个 dataset_path 参数
os.environ["CUSTOM_DATASET_PATH"] = "imf_train.jsonl"

print("Tunix 已准备就绪。请运行下方的训练单元格，它将开始针对 IMF PDF 报告进行精准微调。")


Tunix 已准备就绪。请运行下方的训练单元格，它将开始针对 IMF PDF 报告进行精准微调。


1. 基准测试 (Pre-training Test)：在训练前，我们先问模型一个关于 2024 IMF 报告的深度细节（例如“Policy Pivot”的定义）。此时 2B 模型通常会给出通用回答。
2. 数据适配 (IMF Loader)：我编写了一个新的 IMFDataset适配器，它能直接读取上面生成的 imf_train.jsonl 并将其转化为 Tunix 的训练格式。
3. 极速微调 (Focused SFT)：针对 2B 模型设置了 4-bit QLoRA，旨在 100 步内让它记住 PDF 里的核心知识。
4. 成果验证 (Post-training Inference)：训练结束后，再次运行同一个测试，对比模型在术语理解和文档定位上的变化。

### 1. 微调前：基准测试 (Pre-training Baseline)

运行此单元格，看看原生模型对 2024 年 10 月 IMF 报告的了解程度（通常它会给出模糊的回答）。

In [31]:
# --- 1. 定制金融基准问题 ---
test_questions = [
    "According to the IMF World Economic Outlook October 2024, what is the 'Policy Pivot' and what are the primary risks to global growth?",
    "分析 IMF 2024 年 10 月报告中提到的全球通胀挑战以及其对新兴市场的影响。"
]

# --- 2. 运行微调前推理 ---
import tunix
from tunix.generate import sampler as sampler_lib

# 初始化采样器 (基于训练前的模型状态)
sampler = sampler_lib.Sampler(
    transformer=model,
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=256,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

print("🧐 [基准测试] 微调前原生模型回答:")
out_data_pre = sampler(
    input_strings=test_questions,
    max_generation_steps=150,
    eos_tokens=[tokenizer.eos_id()],
)
for q, a in zip(test_questions, out_data_pre.text):
    print(f"--- Q: {q}\n--- A: {a}\n{'-'*50}")


🧐 [基准测试] 微调前原生模型回答:
--- Q: According to the IMF World Economic Outlook October 2024, what is the 'Policy Pivot' and what are the primary risks to global growth?
--- A: 
--------------------------------------------------
--- Q: 分析 IMF 2024 年 10 月报告中提到的全球通胀挑战以及其对新兴市场的影响。
--- A: 
--------------------------------------------------


### 2. 核心步骤：IMF 数据加载与启动训练 (Loading & Training)

这一步将加载之前生成的 imf_train.jsonl，并由 Tunix 进行针对性的 LoRA 微调。

In [27]:
import json
import numpy as np
from tunix.sft import peft_trainer
from tunix.sft import utils

# --- 1. 防弹版数据加载器（构造实体 List 实装 __len__） ---
class IMFDataLoader:
    def __init__(self, file_path, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.batches = []
        
        # 预先获取 Padding ID
        try:
            pad_id = self.tokenizer.pad_id()
        except:
            pad_id = 0 # 缺省后备保护
            
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                entry = json.loads(line)
                # 拼接标准 Gemma Chat 格式
                full_text = f"<start_of_turn>user\n{entry['instruction']}\n{entry['input']}<end_of_turn>\n"
                full_text += f"<start_of_turn>model\n{entry['output']}<end_of_turn>"
                tokens = tokenizer.encode(full_text)
                
                # 创建静态形状的 Numpy 数组（使用 np 而不是 jnp 防显存溢出）
                arr = np.full((self.max_length,), pad_id, dtype=np.int32)
                ln = min(len(tokens), self.max_length)
                arr[:ln] = tokens[:ln]
                
                # 保持 Batch = 1 的维度
                batched_arr = np.expand_dims(arr, axis=0) 
                # 提前算出 mask，填补 0.1.6 版本的底层深坑
                mask_arr = batched_arr != pad_id
                
                # 封装为标准类，不再使用动态截取
                batch = peft_trainer.TrainingInput(
                    input_tokens=batched_arr,  # 保留在 CPU 内存，JAX 训练步会自己拉取
                    input_mask=mask_arr
                )
                self.batches.append(batch)

    # 支持 Tunix 所有的长度及遍历探测！
    def __len__(self):
        return len(self.batches)
        
    def __iter__(self):
        return iter(self.batches)

# --- 2. 映射函数的兼容包裹 ---
def safe_gen_model_input_fn(x: peft_trainer.TrainingInput):
    # 直接使用我们在 CPU 上算好的 input_mask，防止在 GPU 上进行多余计算
    return {
        'input_tokens': x.input_tokens,
        'input_mask': x.input_mask,
        'positions': utils.build_positions_from_mask(x.input_mask),
        'attention_mask': utils.make_causal_attn_mask(x.input_mask),
    }

# 挂载拦截器
trainer = trainer.with_gen_model_input_fn(safe_gen_model_input_fn)

# 初始化数据集（会直接载入内存转成实体 List）
imf_train_ds = IMFDataLoader("imf_train.jsonl", tokenizer, max_length=1024)
print(f"📦 数据加载完成，共准备好 {len(imf_train_ds)} 组符合 Tunix 协议的训练批次。")

# --- 3. 启动终极训练 ---
print("🚀 启动 IMF 知识注入 (防 OOM 内存管理版)...")
try:
    # 传入空 List 作为空验证集，绝对不会引发 TypeError 或 NoneType 相关错误
    trainer.train(imf_train_ds, [])
    print("✅ 微调成功完成！请前往下一步运行对比基准。")
except Exception as e:
    print("\n💥 如果还是报错了，真正的 Traceback 是：")
    import traceback
    traceback.print_exc()


📦 数据加载完成，共准备好 87 组符合 Tunix 协议的训练批次。
🚀 启动 IMF 知识注入 (防 OOM 内存管理版)...
2026-03-30 00:37:54,889 - Train loop finished in: 0.0001 seconds
2026-03-30 00:37:54,890 - [process=0][thread=MainThread][wait_until_finished] No Save Finalize thread to wait for. Returning.
2026-03-30 00:37:54,891 - Closing _NonBlockingMetadataStore(enable_write=True, _write_lock=<locked _thread.RLock object owner=136089729172096 count=1 at 0x7bc42c596d80>, _store_impl=<orbax.checkpoint._src.metadata.checkpoint._MetadataStoreImpl object at 0x7bc41a263da0>, _single_thread_executor=<concurrent.futures.thread.ThreadPoolExecutor object at 0x7bc41a263e60>, _write_futures=[])
2026-03-30 00:37:54,892 - Closing _NonBlockingMetadataStore(enable_write=True, _write_lock=<locked _thread.RLock object owner=136089729172096 count=1 at 0x7bc42c596d80>, _store_impl=<orbax.checkpoint._src.metadata.checkpoint._MetadataStoreImpl object at 0x7bc41a263da0>, _single_thread_executor=<concurrent.futures.thread.ThreadPoolExecutor object at 0x7b

### 3. 微调后：成果展示 (Post-training Evaluation)

使用完全相同的问题，观察微调后的模型是否能够准确识别并分析 IMF 报告的内容。

In [30]:
# --- 1. 定制金融基准问题 ---
test_questions = [
    "According to the IMF World Economic Outlook October 2024, what is the 'Policy Pivot' and what are the primary risks to global growth?",
    "分析 IMF 2024 年 10 月报告中提到的全球通胀挑战以及其对新兴市场的影响。"
]

# --- 2. [关键修复] 严格使用 Gemma 3 官方指令模板 ---
formatted_questions = []
for q in test_questions:
    # 这是唤醒 Instruct 模型的强制性“暗号”
    prompt = f"<start_of_turn>user\n{q}<end_of_turn>\n<start_of_turn>model\n"
    formatted_questions.append(prompt)


# --- 3. 运行微调后模型推理 ---
print("🧐 正在请求模型生成真实回答...")
out_data_post = sampler_post(
    input_strings=formatted_questions,  # 注意这里传的是包裹好的暗号
    max_generation_steps=150,
    eos_tokens=[tokenizer.eos_id()],
)

print("\n🏆 微调后 IMF 专家模式真实回答:")
for q, a in zip(test_questions, out_data_post.text):
    print(f"--- Q: {q}")
    # 因为生成可能会连带着把一些控制字符打出来，我们可以.strip()清理一下
    print(f"--- A: {a.strip()}")
    print("-" * 50)


🧐 正在请求模型生成真实回答...

🏆 微调后 IMF 专家模式真实回答:
--- Q: According to the IMF World Economic Outlook October 2024, what is the 'Policy Pivot' and what are the primary risks to global growth?
--- A: Selon le rapport de la IMF du monde en économie, quelles sont les "Pivot" de la "Politie" et quelles sont les principales risques pour la croissance mondiale ?
--------------------------------------------------
--- Q: 分析 IMF 2024 年 10 月报告中提到的全球通胀挑战以及其对新兴市场的影响。
--- A: Analyse du 2024 de l'économie des Nations-Unis sur le fait que l'inflation mondiale n'a que peu de temps de chance pour que les marchés émergents n'échouent.
--------------------------------------------------


In [33]:
import os
import shutil
import optax
import jax.numpy as jnp
from tunix.sft import peft_trainer
from flax import nnx
import qwix
from tunix.generate import sampler as sampler_lib

# 🧨 1. 核弹级清理：物理销毁被污染的旧检查点进度！
checkpoint_path = os.path.abspath("./gemma3_lora_results")
if os.path.exists(checkpoint_path):
    print("🗑️ 发现旧的进度缓存，正在强制清理...")
    shutil.rmtree(checkpoint_path)

# 🌱 2. 完全重生：重新切除并复原干净的 LoRA 权重
# (假设 base_model 和 lora_provider 之前已经存在)
print("♻️ 正在为 Base Model 注入全新的纯净 LoRA 矩阵...")
model_input = base_model.get_model_input()
# 这里的 nnx.Rngs(0) 会强制生成初始化的全新权重
clean_model = qwix.apply_lora_to_model(base_model, lora_provider, rngs=nnx.Rngs(0), **model_input)

# 🚀 3. 新建重生的 Trainer
optimizer = optax.adamw(learning_rate=5e-4)
trainer = peft_trainer.PeftTrainer(
    model=clean_model,
    optimizer=optimizer,
    training_config=peft_trainer.TrainingConfig(
        eval_every_n_steps=50,
        max_steps=150,  # 我们只跑 150 步，专攻 IMF 数据
        checkpoint_root_directory=checkpoint_path,
        data_sharding_axis=()
    )
)

# 挂载我们上一个单元格中完美的 Loss Mask 拦截器
trainer = trainer.with_gen_model_input_fn(mask_aware_input_fn)

# ⚔️ 4. 终极一战：启动真实微调
print("🔥 开始强行注入 IMF 知识 (这次不会再跳过了！)...")
trainer.train(new_train_ds, [])


# 🎓 5. 推理大考
print("\n🏆 全新微调后预测：")
tests = [
    "Summarize the key economic metrics in this IMF report excerpt.",
    "What is the 'Policy Pivot' in the October 2024 WEO?"
]
formatted_tests = [f"<start_of_turn>user\n{t}<end_of_turn>\n<start_of_turn>model\n" for t in tests]

# 绑定洗心革面后的 clean_model
sampler_post = sampler_lib.Sampler(
    transformer=clean_model, 
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=256, num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads, head_dim=model_config.head_dim,
    )
)

out = sampler_post(input_strings=formatted_tests, max_generation_steps=120, eos_tokens=[tokenizer.eos_id(), tokenizer.encode("<end_of_turn>")[0]])
for q, a in zip(tests, out.text):
    print(f"\n--- Q: {q}\n--- A: {a.strip()}")


🗑️ 发现旧的进度缓存，正在强制清理...
♻️ 正在为 Base Model 注入全新的纯净 LoRA 矩阵...
2026-03-30 00:47:54,952 - save_device_host_concurrent_bytes=None
2026-03-30 00:47:54,954 - Created BasePyTreeCheckpointHandler: use_ocdbt=True, use_zarr3=False, pytree_metadata_options=PyTreeMetadataOptions(support_rich_types=False), array_metadata_store=<orbax.checkpoint._src.metadata.array_metadata_store.Store object at 0x7bc47e28ee70>, enable_pinned_host_transfer=True, save_concurrent_bytes: 96000000000 (89.4 GiB), restore_concurrent_bytes: 96000000000 (89.4 GiB)
2026-03-30 00:47:54,955 - save_device_host_concurrent_bytes=None
2026-03-30 00:47:54,956 - Created BasePyTreeCheckpointHandler: use_ocdbt=True, use_zarr3=False, pytree_metadata_options=PyTreeMetadataOptions(support_rich_types=False), array_metadata_store=<orbax.checkpoint._src.metadata.array_metadata_store.Store object at 0x7bc47e28ee70>, enable_pinned_host_transfer=True, save_concurrent_bytes: 96000000000 (89.4 GiB), restore_concurrent_bytes: 96000000000 (89.4 GiB)

Training:   0%|          | 0/150 [00:00<?, ?step/s]

2026-03-30 00:48:50,134 - [process=0][thread=MainThread][wait_until_finished] No Save Finalize thread to wait for. Returning.
2026-03-30 00:48:50,135 - [process=0] Saving checkpoint at step 1
2026-03-30 00:48:50,138 - [process=0] Started async saving checkpoint to /content/gemma3_lora_results/1.
2026-03-30 00:48:50,160 - Creating tmp directory /content/gemma3_lora_results/1.orbax-checkpoint-tmp
2026-03-30 00:48:50,214 - Scheduling D2H of 260 prioritized jax.Array.
2026-03-30 00:48:50,215 - Transferring arrays to host memory with options: use_replica_parallel=True, min_slice_bytes_for_replica_parallel=None, max_replicas_for_replica_parallel=None, enable_pinned_host_transfer=True
2026-03-30 00:48:50,226 - Creating tmp directory /content/gemma3_lora_results/1.orbax-checkpoint-tmp/model_params.orbax-checkpoint-tmp
2026-03-30 00:48:50,230 - Creating tmp directory /content/gemma3_lora_results/1.orbax-checkpoint-tmp/optimizer_state.orbax-checkpoint-tmp
2026-03-30 00:48:51,045 - [process=0][th

In [ ]:
# 🔍 暴力终极探针：去掉所有的伪装和暗号，剥离所有复杂的停止符
# 我们要直接看模型最底层的裸金属输出 (Raw Tokens)！！！

print("🧪 探针启动：最原始的输入测试...")

# 极简输入，没有 <start_of_turn>，只给一句话，逼它接话
raw_tests = [
    "The primary risk in the IMF World Economic Outlook October 2024 is",
    "According to the economic excerpt, the core problem discussed is"
]

# 强制重建一个绝对无缓存干扰的极简采样器
probe_sampler = sampler_lib.Sampler(
    transformer=clean_model,   # 使用刚才的微调模型
    tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=128,          # 减小尺寸，加快速度
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    )
)

print("⚖️ 正在进行裸态推理（关闭一切复杂的提早停止符）...")
# 故意不设任何复杂的 eos_tokens！只用系统最致命底层的 eos_id，强行让它生成到 50 步
out_probe = probe_sampler(
    input_strings=raw_tests, 
    max_generation_steps=50, 
    eos_tokens=[tokenizer.eos_id()] 
)

for q, txt, tokens in zip(raw_tests, out_probe.text, out_probe.tokens):
    print(f"\n======== 探针报告 ========")
    print(f"🔸 问题: {q}")
    # 有时候 a.strip() 会把看不见的格式全删了，这次我们暴露它的原貌：
    print(f"🔸 回答: [{txt}]") 
    
    # 终极内窥镜：把它生成的底层机器码 ID 全打印出来！
    # 如果它的 token 列表是空的，说明 JAX 底层断连；
    # 如果全是 [1] (EOS)，说明模型被训成了哑巴；
    # 如果全是 0 或者特殊控制符，说明分词器乱码。
    print(f"🔸 内部 Token ID 序列: \n{tokens.tolist()}")
    print("==========================")
